In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "README.md").exists():
    raise SystemExit("README.md not found above " + str(Path.cwd()) + ". Open the notebook from inside the project folder.")
os.chdir(ROOT)
print("Project folder:", ROOT)


Project folder: /Users/kushalvurukonda/Downloads/dcv-project


In [ ]:
import argparse, glob, os, sys, datetime as dt
import numpy as np
import pandas as pd

TODAY = dt.date.today().isoformat()
WINDOW_LEN = 3
LAST_FULL_YEAR = 2025
LINEAGE_FLAG = 1.05      # lineage count > 5% above own count
OA_FLAG = 1.15           # OpenAlex yearly count > 15% above works-filter count
AIIMS_DELHI_OA = "I63739035"   # looks like a merged all-AIIMS record
N_BOOT = 1000


In [ ]:
def newest(pattern, explicit):
    if explicit:
        return explicit
    files = sorted(glob.glob(pattern))
    if not files:
        sys.exit(f"No file matches {pattern}. Pass the path with the matching option.")
    return files[-1]


In [ ]:
def window_sums(yearly, Y, shift):
    lo, hi = Y - WINDOW_LEN - shift, Y - 1 - shift
    assert hi <= LAST_FULL_YEAR, f"window {lo}-{hi} reaches a partial year"
    sub = yearly[yearly.year.between(lo, hi)]
    g = sub.groupby("openalex_id").agg(
        pubs_own=("works_all_own", "sum"),
        pubs_lineage=("works_all_lineage", "sum"),
        pubs_article_lineage=("works_article_lineage", "sum"),
        cited1_lineage=("works_cited_ge1_lineage", "sum"),
        cited10_lineage=("works_cited_ge10_lineage", "sum"),
        oa_pubs_year=("oa_works_count_year", "sum"),
        cites_received=("oa_cited_by_count_year", "sum"),
        n_oa_years=("year", "nunique"),
        n_oa_missing=("oa_cited_by_count_year", lambda s: int(s.isna().sum())),
    )
    return g


In [ ]:
def spearman(df, a, b):
    d = df[[a, b]].dropna()
    if len(d) < 5:
        return np.nan, len(d)
    return d[a].rank().corr(d[b].rank()), len(d)


In [ ]:
def boot_ci(df, a, b, rng):
    d = df[[a, b]].dropna().reset_index(drop=True)
    n = len(d)
    if n < 10:
        return np.nan, np.nan
    vals = []
    for _ in range(N_BOOT):
        s = d.iloc[rng.integers(0, n, n)]
        r = s[a].rank().corr(s[b].rank())
        if not np.isnan(r):
            vals.append(r)
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


In [ ]:
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--nirf"); ap.add_argument("--yearly"); ap.add_argument("--crosswalk")
    a = ap.parse_args()
    nirf_p = newest("clean/nirf_overall_clean_*.csv", a.nirf)
    year_p = newest("interim/openalex_yearly_*.csv", a.yearly)
    xw_p = newest("interim/openalex_crosswalk_*.csv", a.crosswalk)
    print("NIRF     :", nirf_p); print("YEARLY   :", year_p); print("CROSSWALK:", xw_p)

    nirf = pd.read_csv(nirf_p)
    yearly = pd.read_csv(year_p)
    xw = pd.read_csv(xw_p)
    log = []

    assert xw.id_new.is_unique, "an id_new appears twice in the crosswalk"
    dup_oa = xw[xw.openalex_id.duplicated(keep=False)]
    if len(dup_oa):
        print("WARNING: two NIRF institutions share one OpenAlex id:\n", dup_oa.to_string())
    log.append(("crosswalk rows", len(xw), f"openalex ids shared by 2+ institutions: {dup_oa.openalex_id.nunique()}"))

    n0 = len(nirf)
    df = nirf.merge(xw[["id_new", "openalex_id", "ror_id", "decision"]], on="id_new", how="left", validate="m:1")
    assert len(df) == n0
    no_oa = df.openalex_id.isna().sum()
    log.append(("NIRF rows without an OpenAlex id", int(no_oa), "should be 0"))

    parts = []
    for Y, g in df.groupby("year"):
        main_w = window_sums(yearly, Y, 0)
        sh = window_sums(yearly, Y, 1)[["pubs_own", "cites_received"]].add_suffix("_shift1")
        w = main_w.join(sh, how="left")
        w["year"] = Y
        w["win_from"], w["win_to"] = Y - WINDOW_LEN, Y - 1
        parts.append(w.reset_index())
    wins = pd.concat(parts, ignore_index=True)
    df = df.merge(wins, on=["year", "openalex_id"], how="left", validate="1:1")
    miss = df.pubs_own.isna().sum()
    log.append(("NIRF rows with no OpenAlex window counts", int(miss), "should be 0"))
    log.append(("rows with any missing citation value inside window", int((df.n_oa_missing > 0).sum()), "should be 0"))
    short = df[df.n_oa_years < WINDOW_LEN]
    log.append(("rows whose window has fewer than 3 years of data", len(short), "should be 0"))

    df["pubs_per_year"] = df.pubs_own / WINDOW_LEN
    df["share_ge10_lineage"] = df.cited10_lineage / df.pubs_lineage.replace(0, np.nan)
    df["lineage_ratio"] = df.pubs_lineage / df.pubs_own.replace(0, np.nan)
    df["oa_ratio"] = df.oa_pubs_year / df.pubs_own.replace(0, np.nan)
    df["flag_lineage_inflated"] = df.lineage_ratio > LINEAGE_FLAG
    df["flag_oa_disagree"] = df.oa_ratio > OA_FLAG
    df["flag_aiims_merged"] = df.openalex_id == AIIMS_DELHI_OA
    df["flag_any"] = df[["flag_lineage_inflated", "flag_oa_disagree", "flag_aiims_merged"]].any(axis=1)
    for c in ["flag_lineage_inflated", "flag_oa_disagree", "flag_aiims_merged", "flag_any"]:
        log.append((c, int(df[c].sum()), "institution-years"))

    for col in ["rpc", "pubs_own", "cites_received"]:
        df[f"rank_{col}"] = df.groupby("year")[col].rank(ascending=False, method="min")
    df["rank_gap_pubs_vs_rpc"] = df.rank_pubs_own - df.rank_rpc   # + : publishes less than its research score suggests
    df["rank_gap_cites_vs_rpc"] = df.rank_cites_received - df.rank_rpc

    rng = np.random.default_rng(42)
    rows = []

    def add(year, spec, sub, x, y_, ci=False):
        r, n = spearman(sub, x, y_)
        lo, hi = boot_ci(sub, x, y_, rng) if ci else (np.nan, np.nan)
        rows.append(dict(year=year, spec=spec, metric=y_, n=n, spearman=r, ci_low=lo, ci_high=hi))

    for Y, g in df.groupby("year"):
        for m in ["pubs_own", "cites_received"]:
            add(Y, "main", g, "rpc", m, ci=True)
            add(Y, "excl_flagged", g[~g.flag_any], "rpc", m)
            add(Y, "top50_only", g[g["rank"] <= 50], "rpc", m)
            add(Y, "rank_51_100", g[g["rank"] > 50], "rpc", m)
            add(Y, "shifted_window", g, "rpc", m + "_shift1")
        add(Y, "lineage", g, "rpc", "pubs_lineage")
        add(Y, "lineage", g, "rpc", "share_ge10_lineage")
        add(Y, "overall_rank_control", g.assign(neg_rank=-g["rank"]), "neg_rank", "pubs_own")
    sp = pd.DataFrame(rows)


    dis = (df.assign(abs_gap=df.rank_gap_pubs_vs_rpc.abs())
             .sort_values(["year", "abs_gap"], ascending=[True, False])
             .groupby("year").head(10)
             [["year", "id_new", "name_new", "rank", "rpc", "pubs_own", "rank_rpc", "rank_pubs_own",
               "rank_gap_pubs_vs_rpc", "flag_any"]])

    os.makedirs("clean", exist_ok=True); os.makedirs("interim", exist_ok=True)
    out_join = f"clean/nirf_openalex_joined_{TODAY}.csv"
    out_sp = f"interim/spearman_by_year_{TODAY}.csv"
    out_dis = f"interim/biggest_disagreements_{TODAY}.csv"
    out_log = f"interim/join_log_{TODAY}.csv"
    df.to_csv(out_join, index=False)
    sp.to_csv(out_sp, index=False)
    dis.to_csv(out_dis, index=False)
    pd.DataFrame(log, columns=["step", "count", "detail"]).to_csv(out_log, index=False)

    pd.set_option("display.width", 200)
    print("\nJOIN LOG"); print(pd.DataFrame(log, columns=["step", "count", "detail"]).to_string(index=False))
    print("\nSPEARMAN, NIRF research score (RPC) vs OpenAlex, by year (main specification)")
    mn = sp[sp.spec == "main"].pivot(index="year", columns="metric", values="spearman").round(2)
    print(mn.to_string())
    print("\nSame, by specification (pubs_own only)")
    t = sp[(sp.metric == "pubs_own") | (sp.metric == "pubs_lineage")]
    print(t.pivot_table(index="year", columns="spec", values="spearman").round(2).to_string())
    print(f"\nWrote:\n  {out_join}\n  {out_sp}\n  {out_dis}\n  {out_log}")


In [ ]:
sys.argv = ["notebook"] + []
main()


NIRF     : clean/nirf_overall_clean_2026-11-02.csv
YEARLY   : interim/openalex_yearly_2026-11-02.csv
CROSSWALK: interim/openalex_crosswalk_2026-11-02.csv

JOIN LOG
                                              step  count                                    detail
                                    crosswalk rows    164 openalex ids shared by 2+ institutions: 0
                  NIRF rows without an OpenAlex id      0                               should be 0
          NIRF rows with no OpenAlex window counts      0                               should be 0
rows with any missing citation value inside window      0                               should be 0
  rows whose window has fewer than 3 years of data      0                               should be 0
                             flag_lineage_inflated     88                         institution-years
                                  flag_oa_disagree      4                         institution-years
                                 fla

In [ ]:
import pandas as pd, glob
sp = pd.read_csv(sorted(glob.glob("interim/spearman_by_year_*.csv"))[-1])
sp[sp.spec == "main"].pivot(index="year", columns="metric", values="spearman").round(2)


metric,cites_received,pubs_own
year,,
2017,0.92,0.89
2018,0.89,0.87
2019,0.88,0.78
2020,0.85,0.79
2021,0.84,0.78
2022,0.86,0.77
2023,0.85,0.70
2024,0.87,0.78
2025,0.86,0.72


In [ ]:
sp.pivot_table(index="year", columns="spec", values="spearman", aggfunc="first").round(2)


spec,excl_flagged,lineage,main,overall_rank_control,rank_51_100,shifted_window,top50_only
year,,,,,,,
2017,0.89,0.87,0.89,0.57,0.88,0.90,0.88
2018,0.87,0.85,0.87,0.69,0.76,0.89,0.88
2019,0.78,0.77,0.78,0.68,0.61,0.80,0.80
2020,0.78,0.78,0.79,0.69,0.63,0.78,0.77
2021,0.75,0.77,0.78,0.70,0.63,0.80,0.71
2022,0.73,0.76,0.77,0.71,0.52,0.83,0.73
2023,0.63,0.69,0.70,0.67,0.22,0.76,0.65
2024,0.73,0.77,0.78,0.75,0.44,0.81,0.73
2025,0.68,0.72,0.72,0.65,0.27,0.76,0.67
